# Qwen 2.5-1.5B-Instruct (`qwen-engine`) — Portable Usage Guide

This notebook works anywhere (**Google Colab**, **Kaggle**, **Windows**, or **Linux**). It starts by installing the prebuilt `v0.2.1` release wheel from GitHub.

In [ ]:
import subprocess
import sys

RELEASE_TAG = "v0.2.1"
WHEEL_VER = "0.2.0"
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
base = f"https://github.com/Areen-09/qwen-engine/releases/download/{RELEASE_TAG}"

if sys.platform == "win32" and (3, 9) <= sys.version_info[:2] <= (3, 13):
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}-{py_tag}-{py_tag}-win_amd64.whl"
elif sys.platform.startswith("linux") and (3, 9) <= sys.version_info[:2] <= (3, 13):
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}-{py_tag}-{py_tag}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
else:
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}.tar.gz"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", wheel_url])
import qwen_engine
print("Installed qwen_engine:", qwen_engine.__version__)

## 1. Load Qwen 2.5-1.5B (W4A16 Group-128)
Uses local `models/` if present, otherwise downloads and caches weights automatically.

In [ ]:
from pathlib import Path
from qwen_engine import Engine

if Path("models/qwen25_1.5b_w4a16.qwenbin").is_file():
    engine = Engine("models/qwen25_1.5b_w4a16.qwenbin", "models/tokenizer.bin", device_id=0)
else:
    engine = Engine.from_pretrained("qwen2.5-1.5b-w4a16", device_id=0)

stats = engine.get_memory_stats()
print(f"Weights:   {stats.weights_bytes / 1024**2:.2f} MB")
print(f"KV Cache:  {stats.kv_cache_bytes / 1024**2:.2f} MB")
print(f"Workspace: {stats.workspace_bytes / 1024**2:.2f} MB")
print(f"Total:     {stats.total_allocated_bytes / 1024**2:.2f} MB")

## 2. Streaming ChatML Generation

In [ ]:
import time

engine.reset_session()
messages = [
    {"role": "system", "content": "You are a helpful, concise AI assistant."},
    {"role": "user", "content": "Explain why CUDA Graphs speed up autoregressive LLM decoding in 3 bullet points."}
]

t0 = time.perf_counter()
tokens_out = 0
for chunk in engine.stream(messages, max_tokens=256, temperature=0.0):
    print(chunk, end="", flush=True)
    tokens_out += 1
elapsed = time.perf_counter() - t0
print(f"\n\n--- {tokens_out} tokens in {elapsed:.2f}s ({tokens_out / elapsed:.1f} tok/s) ---")

## 3. Session Rollback & One-Shot Generation

In [ ]:
engine.reset_session()
checkpoint = engine.current_seq_pos
out = engine.generate("List 3 benefits of W4A16 quantization:", max_tokens=96, temperature=0.0)
print(out)

engine.rollback_to(checkpoint)
print(f"\nRolled back KV cache position to {engine.current_seq_pos}")